In [3]:
#Import all libraries and functions.

import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS ,
summarize ,
poly)
from sklearn.model_selection import train_test_split
from functools import partial
from sklearn.model_selection import \
(cross_validate ,
KFold ,
ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

In [5]:
# Load the Auto dataset and split it into training and validation sets.

Auto = load_data('Auto')
Auto_train , Auto_valid = train_test_split (Auto ,
test_size =196,
random_state =0)

In [7]:
# Build and fit a simple linear regression model that predicts mpg from horsepower using the training data.
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train , X_train)
results = model.fit()

In [9]:
#Predict mpg for the validation data and calculate the validation Mean Squared Error (MSE).
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean (( y_valid - valid_pred)**2)

23.61661706966988

In [11]:
# Define a reusable function that fits a regression model on training data and returns its MSE on test data.
def evalMSE(terms ,
response ,
train ,
test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]
    X_test = mm.transform(test)
    y_test = test[response]
    results = sm.OLS(y_train , X_train).fit()
    test_pred = results.predict(X_test)
    return np.mean (( y_test - test_pred)**2)


In [13]:
# Compare polynomial regression models of degrees 1, 2, and 3 using the same training/validation split.
MSE = np.zeros(3)

for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE(
        [poly('horsepower', degree)],
        'mpg',
        Auto_train,
        Auto_valid
    )

MSE

array([23.61661707, 18.76303135, 18.79694163])

In [15]:
#  Repeat the polynomial regression comparison using a different random train/validation split. 
# This demonstrates how the estimated test error can change with the split.

Auto_train, Auto_valid = train_test_split(
    Auto,
    test_size=196,
    random_state=3
)

MSE = np.zeros(3)

for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE(
        [poly('horsepower', degree)],
        'mpg',
        Auto_train,
        Auto_valid
    )

MSE

array([20.75540796, 16.94510676, 16.97437833])

In [17]:
# Create a scikit-learn-compatible OLS model and use Leave-One-Out Cross-Validation (LOOCV) to estimate prediction error for the linear horsepower model.

hp_model = sklearn_sm(sm.OLS ,
MS(['horsepower']))
X, Y = Auto.drop(columns =['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model ,
X,
Y,
cv=Auto.shape [0])
cv_err = np.mean(cv_results['test_score'])
cv_err

24.23151351792922

In [19]:
# Use LOOCV to compare polynomial regression models from degree 1 through degree 5. 

cv_error = np.zeros(5)

H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)

for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    
    M_CV = cross_validate(
        M,
        X,
        Y,
        cv=Auto.shape[0]
    )
    
    cv_error[i] = np.mean(M_CV['test_score'])

cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.4244303 , 19.03322528])

In [21]:
# Demonstrate NumPy's outer addition operation with two small arrays.
A = np.array ([3, 5, 9])
B = np.array ([2, 4])
np.add.outer(A, B)

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

In [23]:
# Use 10-fold cross-validation with shuffled, reproducible folds to compare polynomial models of degrees 1 through 5.

cv_error = np.zeros(5)

cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)  # use same splits for each degree

for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    
    M_CV = cross_validate(
        M,
        X,
        Y,
        cv=cv
    )
    
    cv_error[i] = np.mean(M_CV['test_score'])

cv_error

array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13719048])

In [27]:
# Use one ShuffleSplit validation split to estimate the prediction error of the horsepower model. 
#This is similar to a single train/test split.
validation = ShuffleSplit(n_splits =1,
test_size =196,
random_state =0)
results = cross_validate(hp_model ,
Auto.drop (['mpg'], axis =1),
Auto['mpg'],
cv=validation);
results['test_score']

array([23.61661707])

In [29]:

# Repeat ShuffleSplit validation 10 times and calculate both the mean and standard deviation of the validation errors.

validation = ShuffleSplit(n_splits =10,
test_size =196,
random_state =0)
results = cross_validate(hp_model ,
Auto.drop (['mpg'], axis =1),
Auto['mpg'],
cv=validation)
results['test_score']. mean (), results['test_score'].std()

(23.802232661034164, 1.4218450941091831)

In [31]:
#  Load the Portfolio dataset and define a function that calculates the portfolio allocation parameter alpha from covariance estimates.

Portfolio = load_data('Portfolio')

def alpha_func(D, idx):
    cov_ = np.cov(
        D[['X', 'Y']].loc[idx],
        rowvar=False
    )
    
    return (
        (cov_[1, 1] - cov_[0, 1]) /
        (cov_[0, 0] + cov_[1, 1] - 2 * cov_[0, 1])
    )

In [33]:
# Calculate the alpha estimate using the first 100 observations of the Portfolio dataset.

alpha_func(Portfolio , range (100))

0.57583207459283

In [35]:
# Generate one bootstrap sample of 100 observations with replacement and calculate the alpha estimate for that bootstrap sample.

rng = np.random.default_rng (0)
alpha_func(Portfolio ,
rng.choice (100,
100,
replace=True))

0.6074452469619004

In [37]:
# Define a general bootstrap function that repeatedly samples observations with replacement and 
# estimates the standard error of any supplied statistic.
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    
    n = n or D.shape[0]
    
    for _ in range(B):
        idx = rng.choice(
            D.index,
            n,
            replace=True
        )
        
        value = func(D, idx)
        
        first_ += value
        second_ += value ** 2
    
    return np.sqrt(
        second_ / B - (first_ / B) ** 2
    )

In [39]:
# Use the bootstrap function with 1,000 replications to estimate the standard error of the portfolio alpha estimate.

alpha_SE = boot_SE(alpha_func ,
Portfolio ,
B=1000 ,
seed =0)
alpha_SE

0.09118176521277699

In [41]:
# Define a function that fits an OLS regression model to a bootstrap sample and returns the estimated regression coefficients.

def boot_OLS(model_matrix, response, D, idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    
    return sm.OLS(Y_, X_).fit().params

In [43]:
# Create a specialized bootstrap function for the regression of mpg on horsepower.

hp_func = partial(boot_OLS , MS(['horsepower']), 'mpg')

In [45]:
# Generate 10 bootstrap samples of the Auto dataset and fit the horsepower regression to each sample. 
# The resulting coefficient estimates show bootstrap variability.
rng = np.random.default_rng(0)

np.array([
    boot_OLS(
        MS(['horsepower']),
        'mpg',
        Auto,
        rng.choice(Auto.index, len(Auto), replace=True)
    )
    for _ in range(10)
])

array([[39.12226577, -0.1555926 ],
       [37.18648613, -0.13915813],
       [37.46989244, -0.14112749],
       [38.56723252, -0.14830116],
       [38.95495707, -0.15315141],
       [39.12563927, -0.15261044],
       [38.45763251, -0.14767251],
       [38.43372587, -0.15019447],
       [37.87581142, -0.1409544 ],
       [37.95949036, -0.1451333 ]])

In [47]:
# Use 1,000 bootstrap replications to estimate the standard errors of the intercept and horsepower coefficients.

hp_se = boot_SE(hp_func ,
Auto ,
B=1000 ,
seed =10)
hp_se

intercept     0.731176
horsepower    0.006092
dtype: float64

In [49]:
# Fit the horsepower regression using the full Auto dataset and extract the conventional OLS standard errors 
#for comparison with the bootstrap standard errors.
hp_model.fit(Auto , Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

In [51]:
# Define a quadratic regression model using horsepower and horsepower squared, then estimate
#its coefficient standard errors using 1,000 bootstrap replications.
quad_model = MS([ poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS ,
quad_model ,
'mpg')
boot_SE(quad_func , Auto , B=1000)

intercept                                  1.538641
poly(horsepower, degree=2, raw=True)[0]    0.024696
poly(horsepower, degree=2, raw=True)[1]    0.000090
dtype: float64

In [53]:
M = sm.OLS(Auto['mpg'],
quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64